In [ ]:
# =============================================================================
# TASK 1: DATA PIPELINE (v3 — Final)
# Hierarchical Planning in LLMs — Pass 1
#
# CHANGES FROM v2:
#   - Dataset: hendrycks only (EleutherAI/hendrycks_math), no raw AIME problems
#   - 2 difficulty strata only: hard (level 4-5) and medium (level 3)
#   - 125 problems total: 70 hard + 55 medium
#   - MAX_NEW_TOKENS = 8192 (hard cap, ~5 min max per problem)
#   - Models: Groups 1 and 2 only (5 models), Group 3 dropped
#   - Clean restart: delete Drive dataset_summary.json before running
#
# RUN ORDER (priority):
#   1. DeepSeek-R1-Distill-Llama-8B   (Group 1, reasoning)
#   2. Llama-3.1-8B-Instruct           (Group 1, baseline)
#   3. DeepSeek-R1-Distill-Qwen-7B    (Group 2, medium scale)
#   4. DeepSeek-R1-Distill-Qwen-14B   (Group 2, large)
#   5. DeepSeek-R1-Distill-Qwen-1.5B  (Group 2, small)
#
# OUTPUT: /gdrive/MyDrive/llm_planning_study/traces/{model_short_name}/
#             problem_{id}.json
#             run_log.json
#         /gdrive/MyDrive/llm_planning_study/
#             dataset_summary.json   (shared across all models)
# =============================================================================


# -----------------------------------------------------------------------------
# CELL 1 — Install dependencies
# -----------------------------------------------------------------------------
!pip install transformers accelerate datasets bitsandbytes -q


# -----------------------------------------------------------------------------
# CELL 2 — HuggingFace login (run once per session)
# -----------------------------------------------------------------------------
# from huggingface_hub import login
# login()   # paste your HF token when prompted
#            # required for Llama-3.1-8B-Instruct (gated model)


# -----------------------------------------------------------------------------
# CELL 3 — Imports and config
# -----------------------------------------------------------------------------
import os
import re
import json
import time
import random
import torch
from datetime import datetime
from pathlib import Path
from collections import defaultdict
from datasets import load_dataset, concatenate_datasets
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

# ── MODEL SELECTION ───────────────────────────────────────────────────────────
# Change ONLY this between runs. Everything else is automatic.
#
#   "deepseek-ai/DeepSeek-R1-Distill-Llama-8B"   Group 1 — reasoning   ← start here
#   "meta-llama/Llama-3.1-8B-Instruct"            Group 1 — baseline
#   "deepseek-ai/DeepSeek-R1-Distill-Qwen-7B"    Group 2 — medium
#   "deepseek-ai/DeepSeek-R1-Distill-Qwen-14B"   Group 2 — large
#   "deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B"  Group 2 — small

#MODEL_NAME = "deepseek-ai/DeepSeek-R1-Distill-Llama-8B"
#MODEL_NAME = "meta-llama/Llama-3.1-8B-Instruct"
#MODEL_NAME = "deepseek-ai/DeepSeek-R1-Distill-Qwen-7B"
#MODEL_NAME = "deepseek-ai/DeepSeek-R1-Distill-Qwen-14B"
MODEL_NAME = "deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B"
# ── GENERATION CONFIG ─────────────────────────────────────────────────────────
MAX_NEW_TOKENS = 8192    # hard cap ~5 min per problem; truncated traces dropped at analysis
DO_SAMPLE      = False   # greedy decoding — deterministic, reproducible

# ── DATASET CONFIG ────────────────────────────────────────────────────────────
N_HARD    = 200    # hendrycks level 4-5 (hard, AIME-equivalent difficulty)
N_MEDIUM  = 100    # hendrycks level 3   (medium, AMC-equivalent difficulty)
RANDOM_SEED = 42  # fixed for reproducibility across all model runs

# ── HENDRYCKS SUBJECT CONFIGS ─────────────────────────────────────────────────
HENDRYCKS_CONFIGS = [
    "algebra",
    "counting_and_probability",
    "geometry",
    "intermediate_algebra",
    "number_theory",
    "prealgebra",
    "precalculus",
]

# ── QUANTIZATION ──────────────────────────────────────────────────────────────
USE_4BIT = "32B" in MODEL_NAME or "32b" in MODEL_NAME

# ── PATHS ─────────────────────────────────────────────────────────────────────
GDRIVE_BASE    = Path("/content/gdrive/MyDrive/llm_planning_study")
MODEL_SHORT    = MODEL_NAME.split("/")[-1]
TRACE_DIR      = GDRIVE_BASE / "traces" / MODEL_SHORT
LOG_FILE       = TRACE_DIR / "run_log.json"
SUMMARY_FILE   = GDRIVE_BASE / "dataset_summary.json"

print(f"Model:       {MODEL_NAME}")
print(f"Short name:  {MODEL_SHORT}")
print(f"Output dir:  {TRACE_DIR}")
print(f"Use 4-bit:   {USE_4BIT}")
print(f"Max tokens:  {MAX_NEW_TOKENS}")
print(f"Target:      {N_HARD} hard + {N_MEDIUM} medium = {N_HARD + N_MEDIUM} problems")


# -----------------------------------------------------------------------------
# CELL 4 — Mount Google Drive
# -----------------------------------------------------------------------------
from google.colab import drive
drive.mount("/content/gdrive")
TRACE_DIR.mkdir(parents=True, exist_ok=True)
GDRIVE_BASE.mkdir(parents=True, exist_ok=True)
print(f"Drive mounted. Trace dir ready: {TRACE_DIR}")


# -----------------------------------------------------------------------------
# CELL 5 — Load and sample problems from EleutherAI/hendrycks_math
# -----------------------------------------------------------------------------
def load_problems(n_hard: int, n_medium: int, seed: int) -> list[dict]:
    """
    Load from EleutherAI/hendrycks_math across all subject configs.
    Stratify by level:
        level 4-5 → "hard"    (70 problems)
        level 3   → "medium"  (55 problems)
        level 1-2 → dropped   (too easy, insufficient revisions expected)

    Returns problems sorted: hard first, then medium.
    """
    print("Loading EleutherAI/hendrycks_math...")
    all_splits = []
    for config in HENDRYCKS_CONFIGS:
        try:
            ds = load_dataset(
                "EleutherAI/hendrycks_math",
                config,
                split="train",
                trust_remote_code=False
            )
            all_splits.append(ds)
            print(f"  {config:30s}: {len(ds)} problems")
        except Exception as e:
            print(f"  WARNING — could not load '{config}': {e}")

    if not all_splits:
        raise RuntimeError("Could not load any hendrycks configs. Check HF login.")

    full_ds = concatenate_datasets(all_splits)
    print(f"\nTotal loaded: {len(full_ds)} problems across all subjects")

    # Separate into difficulty pools
    hard_pool   = []
    medium_pool = []

    boxed_re = re.compile(r'\\boxed\{([^}]*)\}')

    for i, row in enumerate(full_ds):
        # Parse "Level N" → integer
        level_str = str(row.get("level", "Level 3"))
        try:
            level = int(re.search(r'\d', level_str).group())
        except (AttributeError, ValueError):
            level = 3

        # Skip levels 1-2
        if level < 3:
            continue

        # Extract answer from last \boxed{} in solution
        solution = row.get("solution", "")
        matches  = boxed_re.findall(solution)
        answer   = matches[-1].strip() if matches else ""

        entry = {
            "problem":  row["problem"],
            "answer":   answer,
            "solution": solution,       # keep for answer verification
            "source":   "EleutherAI/hendrycks_math",
            "subject":  row.get("type", "unknown"),
            "level":    level,
            "year":     "various",
        }

        if level >= 4:
            entry["problem_id"]  = f"hard_{i:05d}"
            entry["difficulty"]  = "hard"
            hard_pool.append(entry)
        else:  # level == 3
            entry["problem_id"]  = f"med_{i:05d}"
            entry["difficulty"]  = "medium"
            medium_pool.append(entry)

    print(f"\nPool sizes:")
    print(f"  hard   (level 4-5): {len(hard_pool)}")
    print(f"  medium (level 3):   {len(medium_pool)}")

    # Sample with fixed seed
    rng = random.Random(seed)

    def safe_sample(pool, n, label):
        if len(pool) < n:
            print(f"  WARNING: only {len(pool)} {label} problems available "
                  f"(wanted {n}). Using all.")
            return list(pool)
        return rng.sample(pool, n)

    hard_sample   = safe_sample(hard_pool,   n_hard,   "hard")
    medium_sample = safe_sample(medium_pool, n_medium, "medium")

    # Sort: hard first, then medium (for logical progress display)
    problems = hard_sample + medium_sample

    print(f"\nFinal dataset: {len(problems)} problems")
    print(f"  hard:   {len(hard_sample)}")
    print(f"  medium: {len(medium_sample)}")

    # Subject distribution (informational)
    subject_counts = defaultdict(int)
    for p in problems:
        subject_counts[p["subject"]] += 1
    print("\nSubject distribution:")
    for subj, cnt in sorted(subject_counts.items(), key=lambda x: -x[1]):
        print(f"  {subj:35s}: {cnt}")

    return problems


# -----------------------------------------------------------------------------
# CELL 6 — Dataset summary (shared across all 5 model runs)
# -----------------------------------------------------------------------------
def get_or_create_problems() -> list[dict]:
    """
    If dataset_summary.json already exists on Drive, load that exact problem
    order (ensures all 5 models see identical problems).
    Otherwise, sample fresh and save.
    """
    if SUMMARY_FILE.exists():
        print(f"Loading existing dataset summary from Drive...")
        with open(SUMMARY_FILE) as f:
            summary = json.load(f)

        # Rebuild full problem list from summary
        # (summary stores metadata only; we need to reload problem text)
        problems_fresh = load_problems(N_HARD, N_MEDIUM, RANDOM_SEED)
        id_to_problem  = {p["problem_id"]: p for p in problems_fresh}
        ordered_ids    = [p["problem_id"] for p in summary["problems"]]
        problems       = [id_to_problem[pid] for pid in ordered_ids
                          if pid in id_to_problem]

        print(f"Restored {len(problems)} problems in original order.")
        return problems

    else:
        print("No existing summary — sampling fresh dataset...")
        problems = load_problems(N_HARD, N_MEDIUM, RANDOM_SEED)

        # Save summary
        summary = {
            "created":      datetime.now().isoformat(),
            "n_problems":   len(problems),
            "n_hard":       sum(1 for p in problems if p["difficulty"] == "hard"),
            "n_medium":     sum(1 for p in problems if p["difficulty"] == "medium"),
            "max_new_tokens": MAX_NEW_TOKENS,
            "random_seed":  RANDOM_SEED,
            "problems": [
                {
                    "problem_id": p["problem_id"],
                    "difficulty": p["difficulty"],
                    "level":      p["level"],
                    "subject":    p["subject"],
                    "answer":     p["answer"],
                }
                for p in problems
            ]
        }
        with open(SUMMARY_FILE, "w") as f:
            json.dump(summary, f, indent=2)
        print(f"Dataset summary saved to: {SUMMARY_FILE}")
        return problems


problems = get_or_create_problems()


# -----------------------------------------------------------------------------
# CELL 7 — Checkpoint log
# -----------------------------------------------------------------------------
def load_run_log() -> dict:
    if LOG_FILE.exists():
        with open(LOG_FILE) as f:
            log = json.load(f)
        n_done   = len(log["completed"])
        n_failed = len(log["failed"])
        print(f"Resuming. Completed: {n_done} | Failed: {n_failed} | "
              f"Remaining: {len(problems) - n_done - n_failed}")
    else:
        log = {
            "model":     MODEL_NAME,
            "started":   datetime.now().isoformat(),
            "completed": [],
            "failed":    [],
        }
        print(f"Fresh run. {len(problems)} problems to generate.")
    return log


def save_run_log(log: dict):
    with open(LOG_FILE, "w") as f:
        json.dump(log, f, indent=2)


run_log   = load_run_log()
completed = set(run_log["completed"])
remaining = [p for p in problems if p["problem_id"] not in completed]
print(f"Problems remaining this session: {len(remaining)}")


# -----------------------------------------------------------------------------
# CELL 8 — Load model
# -----------------------------------------------------------------------------
def load_model_and_tokenizer(model_name: str, use_4bit: bool):
    print(f"\nLoading tokenizer: {model_name}")
    tokenizer = AutoTokenizer.from_pretrained(
        model_name,
        trust_remote_code=True
    )
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    print(f"Loading model ({'4-bit' if use_4bit else 'fp16'})...")
    if use_4bit:
        bnb_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_compute_dtype=torch.float16,
            bnb_4bit_use_double_quant=True,
            bnb_4bit_quant_type="nf4"
        )
        model = AutoModelForCausalLM.from_pretrained(
            model_name,
            quantization_config=bnb_config,
            device_map="auto",
            trust_remote_code=True
        )
    else:
        model = AutoModelForCausalLM.from_pretrained(
            model_name,
            torch_dtype=torch.float16,
            device_map="auto",
            trust_remote_code=True
        )

    model.eval()
    print("Model loaded.")
    return model, tokenizer


model, tokenizer = load_model_and_tokenizer(MODEL_NAME, USE_4BIT)


# -----------------------------------------------------------------------------
# CELL 9 — Prompt builder
# -----------------------------------------------------------------------------
def build_prompt(problem: dict, tokenizer, model_name: str) -> str:
    is_reasoning = any(tag in model_name for tag in ["R1", "QwQ"])

    if problem["difficulty"] == "hard":
        instruction = (
            "Solve the following competition mathematics problem carefully. "
            "Show your reasoning step by step and verify your answer."
        )
    else:
        instruction = (
            "Solve the following mathematics competition problem. "
            "Show your working step by step."
        )

    if is_reasoning:
        messages = [{"role": "user",
                     "content": f"{instruction}\n\nProblem: {problem['problem']}"}]
    else:
        messages = [
            {"role": "system",
             "content": "You are an expert mathematician. Solve problems with "
                        "clear step-by-step reasoning."},
            {"role": "user",
             "content": f"{instruction}\n\nProblem: {problem['problem']}"}
        ]

    return tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )


# -----------------------------------------------------------------------------
# CELL 10 — Answer extraction
# -----------------------------------------------------------------------------
def extract_answer(text: str) -> str:
    # Last \boxed{} is most reliable
    boxed = re.findall(r'\\boxed\{([^}]+)\}', text)
    if boxed:
        return boxed[-1].strip()
    # Fallback: "the answer is X"
    m = re.search(r'(?:answer is|therefore)[:\s]*([0-9A-Za-z\-\.\/]+)',
                  text, re.IGNORECASE)
    return m.group(1).strip() if m else "UNKNOWN"


# -----------------------------------------------------------------------------
# CELL 11 — Generation function
# -----------------------------------------------------------------------------
@torch.no_grad()
def generate_trace(problem: dict) -> dict:
    prompt     = build_prompt(problem, tokenizer, MODEL_NAME)
    inputs     = tokenizer(prompt, return_tensors="pt",
                           truncation=False).to(model.device)
    prompt_len = inputs["input_ids"].shape[1]

    t0 = time.time()
    output_ids = model.generate(
        **inputs,
        max_new_tokens=MAX_NEW_TOKENS,
        do_sample=False,
        temperature=None,
        top_p=None,
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=tokenizer.eos_token_id,
    )
    elapsed = time.time() - t0

    new_tokens     = output_ids[0][prompt_len:]
    full_response  = tokenizer.decode(new_tokens, skip_special_tokens=False)
    response_clean = tokenizer.decode(new_tokens, skip_special_tokens=True)

    # Extract <think> block (R1/QwQ models)
    think_content = ""
    if "<think>" in full_response and "</think>" in full_response:
        start = full_response.index("<think>") + len("<think>")
        end   = full_response.index("</think>")
        think_content = full_response[start:end].strip()

    analysis_text = think_content if think_content else response_clean
    model_answer  = extract_answer(analysis_text)
    truncated     = len(new_tokens) >= MAX_NEW_TOKENS - 10

    return {
        # Identity
        "problem_id":        problem["problem_id"],
        "model":             MODEL_NAME,
        # Problem metadata
        "problem":           problem["problem"],
        "answer_gt":         problem["answer"],
        "difficulty":        problem["difficulty"],
        "level":             problem["level"],
        "subject":           problem["subject"],
        # Token counts
        "prompt_tokens":     prompt_len,
        "response_tokens":   len(new_tokens),
        # Text
        "full_response":     full_response,
        "response_clean":    response_clean,
        "think_content":     think_content,
        # Answer
        "model_answer":      model_answer,
        "correct":           model_answer.strip() == problem["answer"].strip(),
        # Metadata
        "generation_time_s": round(elapsed, 2),
        "timestamp":         datetime.now().isoformat(),
        "max_new_tokens":    MAX_NEW_TOKENS,
        "truncated":         truncated,
    }


# -----------------------------------------------------------------------------
# CELL 12 — Main generation loop
# -----------------------------------------------------------------------------
def save_trace(trace: dict):
    out = TRACE_DIR / f"{trace['problem_id']}.json"
    with open(out, "w") as f:
        json.dump(trace, f, indent=2, ensure_ascii=False)


# Track per-difficulty progress
diff_done  = defaultdict(int)
diff_total = defaultdict(int)
for p in problems:
    diff_total[p["difficulty"]] += 1
for p in problems:
    if p["problem_id"] in completed:
        diff_done[p["difficulty"]] += 1

print(f"\nStarting generation: {len(remaining)} problems")
print(f"Model: {MODEL_NAME}")
print("=" * 65)

for i, problem in enumerate(remaining):
    pid  = problem["problem_id"]
    diff = problem["difficulty"]

    h_prog = f"{diff_done['hard']}/{diff_total['hard']}"
    m_prog = f"{diff_done['medium']}/{diff_total['medium']}"

    print(f"\n[{i+1}/{len(remaining)}] {pid}  "
          f"diff={diff}  hard:{h_prog}  med:{m_prog}")

    try:
        trace = generate_trace(problem)
        save_trace(trace)

        run_log["completed"].append(pid)
        save_run_log(run_log)
        diff_done[diff] += 1

        trunc_flag = " [TRUNCATED — will skip in analysis]" if trace["truncated"] else ""
        correct    = "✓" if trace["correct"] else "✗"
        print(f"  {correct} ans={trace['model_answer']} (gt={trace['answer_gt']}) "
              f"| {trace['response_tokens']} tok "
              f"| {trace['generation_time_s']}s{trunc_flag}")

        torch.cuda.empty_cache()

    except Exception as e:
        print(f"  FAILED: {e}")
        run_log["failed"].append({"problem_id": pid, "error": str(e)})
        save_run_log(run_log)

print("\n" + "=" * 65)
print(f"Session complete. Completed: {len(run_log['completed'])} | "
      f"Failed: {len(run_log['failed'])}")


# -----------------------------------------------------------------------------
# CELL 13 — Pilot analysis (run after first model completes)
# Validates truncation rate and storage estimates before running remaining models
# -----------------------------------------------------------------------------
REVISION_RE = re.compile(
    r"\bwait\b|\bactually\b|let me reconsider|that'?s wrong|"
    r"i made an error|let me try again|this doesn'?t work|"
    r"i need to reconsider|\bhmm\b|that'?s not right|"
    r"let me rethink|on second thought|let me go back|"
    r"wait,? let me|actually,? let me|let me re-?check",
    re.IGNORECASE
)


def pilot_analysis():
    files = [f for f in TRACE_DIR.glob("*.json") if f.name != "run_log.json"]
    if not files:
        print("No traces yet.")
        return

    by_diff = defaultdict(lambda: {
        "n": 0, "tokens": [], "revisions": [], "truncated": 0, "correct": 0
    })

    for f in files:
        with open(f) as fp:
            t = json.load(fp)
        d    = t.get("difficulty", "unknown")
        text = t.get("think_content") or t.get("response_clean", "")
        # ── FIX: clean tokenizer artifacts before regex ──────────────────
        text = text.replace("Ġ", " ").replace("Ċ", "\n").replace("ĉ", "\t")
        # ─────────────────────────────────────────────────────────────────
        revs = len(REVISION_RE.findall(text))

        by_diff[d]["n"] += 1
        by_diff[d]["tokens"].append(t["response_tokens"])
        by_diff[d]["revisions"].append(revs)
        if t.get("truncated"):     by_diff[d]["truncated"] += 1
        if t.get("correct"):       by_diff[d]["correct"]   += 1

    print("\n" + "=" * 65)
    print("PILOT ANALYSIS")
    print("=" * 65)

    total_gb = 0
    for diff in ["hard", "medium"]:
        if diff not in by_diff:
            continue
        d         = by_diff[diff]
        n         = d["n"]
        avg_tok   = sum(d["tokens"]) / n
        avg_rev   = sum(d["revisions"]) / n
        trunc_pct = 100 * d["truncated"] / n
        acc       = 100 * d["correct"] / n

        n_layers, hidden, fp16 = 32, 4096, 2
        window   = 60    # ── FIX: updated to ±30 tokens ──────────────────
        full_n   = (N_HARD if diff == "hard" else N_MEDIUM)
        est_gb   = (n_layers * hidden * fp16 * window * avg_rev * full_n) / 1024**3
        total_gb += est_gb

        print(f"\n  {diff.upper()}")
        print(f"    Traces so far:   {n}")
        print(f"    Avg tokens:      {avg_tok:.0f}")
        print(f"    Avg revisions:   {avg_rev:.1f}")
        print(f"    Accuracy:        {acc:.1f}%")
        print(f"    Truncated:       {trunc_pct:.1f}%")
        print(f"    Est. storage:    {est_gb:.2f} GB (for {full_n} problems)")

    print(f"\n  TOTAL est. storage per model: {total_gb:.2f} GB")
    print(f"  TOTAL est. storage (5 models): {total_gb * 5:.1f} GB")
    print()
    if by_diff["hard"]["truncated"] / max(by_diff["hard"]["n"], 1) > 0.3:
        print("  ⚠ WARNING: >30% truncation on hard problems.")
        print("    Consider dropping level-5 problems or reducing N_HARD.")
    else:
        print("  ✓ Truncation rate looks acceptable. Proceed with remaining models.")



   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 43.1 MB/s eta 0:00:00
Model:       deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B
Short name:  DeepSeek-R1-Distill-Qwen-1.5B
Output dir:  /content/gdrive/MyDrive/llm_planning_study/traces/DeepSeek-R1-Distill-Qwen-1.5B
Use 4-bit:   False
Max tokens:  8192
Target:      200 hard + 100 medium = 300 problems
Mounted at /content/gdrive
Drive mounted. Trace dir ready: /content/gdrive/MyDrive/llm_planning_study/traces/DeepSeek-R1-Distill-Qwen-1.5B
Loading existing dataset summary from Drive...
Loading EleutherAI/hendrycks_math...


README.md:   0%|          | 0.00/3.93k [00:00<?, ?B/s]

algebra/train-00000-of-00001.parquet:   0%|          | 0.00/505k [00:00<?, ?B/s]

algebra/test-00000-of-00001.parquet:   0%|          | 0.00/353k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/1744 [00:00<?, ? examples/s]

...

model.safetensors:   0%|          | 0.00/3.55G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/181 [00:00<?, ?B/s]

Model loaded.

Starting generation: 0 problems
Model: deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B

Session complete. Completed: 300 | Failed: 0


KeyError: 'response_tokens'

In [ ]:
import json
from pathlib import Path
from collections import defaultdict
from datetime import datetime

# Assuming REVISION_RE, MODEL_SHORT, N_HARD, N_MEDIUM, GDRIVE_BASE are defined globally
# from the main script (P7nmYoxkilG9).
# If not, you might need to re-import/define them here for this cell to run standalone.

def save_pilot_analysis():
    # Exclude run_log.json and pilot_analysis.json themselves
    files = [f for f in TRACE_DIR.glob("*.json") if f.name not in ["run_log.json", "pilot_analysis.json"]]
    if not files:
        print("No traces to analyze.")
        return

    by_diff = defaultdict(lambda: {
        "n": 0, "tokens": [], "revisions": [], "truncated": 0, "correct": 0
    })

    for f in files:
        try:
            with open(f) as fp:
                t = json.load(fp)
        except json.JSONDecodeError:
            print(f"WARNING: Skipping malformed JSON file: {f.name}")
            continue

        d    = t.get("difficulty", "unknown")
        text = t.get("think_content") or t.get("response_clean", "")
        text = text.replace("Ġ", " ").replace("Ċ", "\n").replace("ĉ", "\t")
        revs = len(REVISION_RE.findall(text))

        # Safely access keys with .get() or check for existence
        response_tokens = t.get("response_tokens")
        if response_tokens is None:
            print(f"WARNING: Trace file {f.name} is missing 'response_tokens'. Skipping this trace's token count.")
            continue # Or handle as 0 tokens if preferred

        by_diff[d]["n"] += 1
        by_diff[d]["tokens"].append(response_tokens)
        by_diff[d]["revisions"].append(revs)
        if t.get("truncated"): by_diff[d]["truncated"] += 1
        if t.get("correct"):   by_diff[d]["correct"]   += 1

    results = {
        "model":       MODEL_SHORT,
        "timestamp":   datetime.now().isoformat(),
        "n_hard":      N_HARD,
        "n_medium":    N_MEDIUM,
        "window_tokens": 60,
        "by_difficulty": {}
    }

    for diff in ["hard", "medium"]:
        if diff not in by_diff or by_diff[diff]["n"] == 0:
            continue
        d  = by_diff[diff]
        n  = d["n"]

        avg_tok  = sum(d["tokens"]) / n if d["tokens"] else 0
        avg_rev  = sum(d["revisions"]) / n if d["revisions"] else 0
        est_gb   = (32 * 4096 * 2 * 60 * avg_rev * \
                   (N_HARD if diff == "hard" else N_MEDIUM)) / 1024**3

        results["by_difficulty"][diff] = {
            "n_traces":       n,
            "avg_tokens":     round(avg_tok, 1),
            "min_tokens":     min(d["tokens"]) if d["tokens"] else 0,
            "max_tokens":     max(d["tokens"]) if d["tokens"] else 0,
            "avg_revisions":  round(avg_rev, 1),
            "min_revisions":  min(d["revisions"]) if d["revisions"] else 0,
            "max_revisions":  max(d["revisions"]) if d["revisions"] else 0,
            "pct_truncated":  round(100 * d["truncated"] / n, 1),
            "accuracy":       round(100 * d["correct"] / n, 1),
            "est_storage_gb": round(est_gb, 2),
        }

    # Save per-model file
    model_pilot = TRACE_DIR / "pilot_analysis.json"
    with open(model_pilot, "w") as f:
        json.dump(results, f, indent=2)
    print(f"  Saved: {model_pilot}")

    # Append to cross-model summary
    cross_file = GDRIVE_BASE / "pilot_summary_all_models.json"
    if cross_file.exists():
        with open(cross_file) as f:
            cross = json.load(f)
    else:
        cross = {}
    cross[MODEL_SHORT] = results
    with open(cross_file, "w") as f:
        json.dump(cross, f, indent=2)
    print(f"  Cross-model summary updated: {cross_file}")

    return results


# Run after generation completes
save_pilot_analysis()


In [ ]:
import json
from pathlib import Path

# Using the TRACE_DIR for the currently active model from the main notebook
TRACE_DIR = Path(f"/content/gdrive/MyDrive/llm_planning_study/traces/{MODEL_SHORT}")

trace_files = [f for f in TRACE_DIR.glob("*.json") if f.name != "run_log.json" and f.name != "pilot_analysis.json"]

if trace_files:
    # Load the first trace file found
    first_trace_file = trace_files[0]
    with open(first_trace_file, "r") as f:
        first_trace_data = json.load(f)

    print(f"Keys of a sample trace file ({first_trace_file.name}):")
    for key in first_trace_data.keys():
        print(f"- {key}")
else:
    print(f"No trace files found in {TRACE_DIR}. Please run the generation loop first.")

In [ ]:
from google.colab import runtime
runtime.unassign()


In [ ]:
import json
from pathlib import Path
from collections import defaultdict

trace_dir = Path("/content/gdrive/MyDrive/llm_planning_study/traces/DeepSeek-R1-Distill-Llama-8B")
files = [f for f in trace_dir.glob("*.json") if f.name != "run_log.json"]

stats = defaultdict(lambda: {"total": 0, "truncated": 0, "tokens": []})

for f in files:
    with open(f) as fp:
        t = json.load(fp)
    d = t.get("difficulty", "unknown")
    stats[d]["total"] += 1
    stats[d]["tokens"].append(t["response_tokens"])
    if t.get("truncated"):
        stats[d]["truncated"] += 1

for diff, s in stats.items():
    avg = sum(s["tokens"]) / len(s["tokens"])
    trunc_pct = 100 * s["truncated"] / s["total"]
    print(f"{diff:15s}: {s['total']} problems, "
          f"{trunc_pct:.0f}% truncated, avg {avg:.0f} tokens")

In [ ]:
from pathlib import Path
import json, re
from collections import defaultdict
from google.colab import drive
drive.mount("/content/gdrive")
# Point to the R1-Llama traces
TRACE_DIR = Path("/content/gdrive/MyDrive/llm_planning_study/traces/DeepSeek-R1-Distill-Llama-8B")

REVISION_RE = re.compile(
    r"\bwait\b|\bactually\b|let me reconsider|that'?s wrong|"
    r"i made an error|let me try again|this doesn'?t work|"
    r"i need to reconsider|\bhmm\b|that'?s not right|"
    r"let me rethink|on second thought|let me go back|"
    r"wait,? let me|actually,? let me|let me re-?check",
    re.IGNORECASE
)

files = [f for f in TRACE_DIR.glob("*.json") if f.name != "run_log.json"]
print(f"Found {len(files)} trace files")

by_diff = defaultdict(lambda: {
    "n": 0, "tokens": [], "revisions": [], "truncated": 0, "correct": 0
})

for f in files:
    with open(f) as fp:
        t = json.load(fp)
    d    = t.get("difficulty", "unknown")
    text = t.get("think_content") or t.get("response_clean", "")
    revs = len(REVISION_RE.findall(text))

    by_diff[d]["n"] += 1
    by_diff[d]["tokens"].append(t["response_tokens"])
    by_diff[d]["revisions"].append(revs)
    if t.get("truncated"): by_diff[d]["truncated"] += 1
    if t.get("correct"):   by_diff[d]["correct"]   += 1

for diff in ["hard", "medium"]:
    if diff not in by_diff:
        print(f"\n  {diff}: no traces found")
        continue
    d         = by_diff[diff]
    n         = d["n"]
    avg_tok   = sum(d["tokens"]) / n
    avg_rev   = sum(d["revisions"]) / n
    trunc_pct = 100 * d["truncated"] / n
    acc       = 100 * d["correct"] / n
    print(f"\n  {diff.upper()}")
    print(f"    Traces:        {n}")
    print(f"    Avg tokens:    {avg_tok:.0f}")
    print(f"    Avg revisions: {avg_rev:.1f}")
    print(f"    Accuracy:      {acc:.1f}%")
    print(f"    Truncated:     {trunc_pct:.1f}%")

In [ ]:
# Check what the think_content actually looks like
import json
from pathlib import Path

TRACE_DIR = Path("/content/gdrive/MyDrive/llm_planning_study/traces/DeepSeek-R1-Distill-Llama-8B")

files = [f for f in TRACE_DIR.glob("*.json") if f.name != "run_log.json"]

with open(files[0]) as f:
    t = json.load(f)

print("=== think_content (first 500 chars) ===")
print(repr(t.get("think_content", "")[:500]))
print()
print("=== response_clean (first 500 chars) ===")
print(repr(t.get("response_clean", "")[:500]))
print()
print(f"think_content length: {len(t.get('think_content', ''))}")
print(f"response_clean length: {len(t.get('response_clean', ''))}")

In [ ]:
import json, re
from pathlib import Path

def clean_tokenizer_artifacts(text: str) -> str:
    text = text.replace("Ġ", " ")
    text = text.replace("Ċ", "\n")
    text = text.replace("ĉ", "\t")
    return text

REVISION_RE = re.compile(
    r"\bwait\b|\bactually\b|let me reconsider|that'?s wrong|"
    r"i made an error|let me try again|this doesn'?t work|"
    r"i need to reconsider|\bhmm\b|that'?s not right|"
    r"let me rethink|on second thought|let me go back|"
    r"wait,? let me|actually,? let me|let me re-?check",
    re.IGNORECASE
)

TRACE_DIR = Path("/content/gdrive/MyDrive/llm_planning_study/traces/DeepSeek-R1-Distill-Llama-8B")
files = [f for f in TRACE_DIR.glob("*.json") if f.name != "run_log.json"]

# Test on first 5 traces
for f in files[:5]:
    with open(f) as fp:
        t = json.load(fp)

    raw_text     = t.get("think_content") or t.get("response_clean", "")
    cleaned_text = clean_tokenizer_artifacts(raw_text)

    raw_revs     = len(REVISION_RE.findall(raw_text))
    cleaned_revs = len(REVISION_RE.findall(cleaned_text))

    print(f"{t['problem_id']} | {t['difficulty']} | "
          f"raw_revisions={raw_revs} | cleaned_revisions={cleaned_revs} | "
          f"tokens={t['response_tokens']}")

In [ ]:
from pathlib import Path
import json, re
from collections import defaultdict

def clean_tokenizer_artifacts(text: str) -> str:
    text = text.replace("Ġ", " ")
    text = text.replace("Ċ", "\n")
    text = text.replace("ĉ", "\t")
    return text

REVISION_RE = re.compile(
    r"\bwait\b|\bactually\b|let me reconsider|that'?s wrong|"
    r"i made an error|let me try again|this doesn'?t work|"
    r"i need to reconsider|\bhmm\b|that'?s not right|"
    r"let me rethink|on second thought|let me go back|"
    r"wait,? let me|actually,? let me|let me re-?check",
    re.IGNORECASE
)

# Run for BOTH models
for model_short in [
    "DeepSeek-R1-Distill-Llama-8B",
    "Llama-3.1-8B-Instruct"
]:
    TRACE_DIR = Path(f"/content/gdrive/MyDrive/llm_planning_study/traces/{model_short}")
    files = [f for f in TRACE_DIR.glob("*.json") if f.name != "run_log.json"]

    if not files:
        print(f"\n{model_short}: no traces found, skipping.")
        continue

    by_diff = defaultdict(lambda: {
        "n": 0, "tokens": [], "revisions": [], "truncated": 0, "correct": 0
    })

    for fp in files:
        with open(fp) as f:
            t = json.load(f)
        d    = t.get("difficulty", "unknown")
        text = t.get("think_content") or t.get("response_clean", "")
        text = clean_tokenizer_artifacts(text)
        revs = len(REVISION_RE.findall(text))

        by_diff[d]["n"] += 1
        by_diff[d]["tokens"].append(t["response_tokens"])
        by_diff[d]["revisions"].append(revs)
        if t.get("truncated"): by_diff[d]["truncated"] += 1
        if t.get("correct"):   by_diff[d]["correct"]   += 1

    print(f"\n{'='*55}")
    print(f"MODEL: {model_short}")
    print(f"{'='*55}")

    total_gb = 0
    for diff in ["hard", "medium"]:
        if diff not in by_diff:
            print(f"  {diff}: no traces")
            continue
        d         = by_diff[diff]
        n         = d["n"]
        avg_tok   = sum(d["tokens"]) / n
        avg_rev   = sum(d["revisions"]) / n
        trunc_pct = 100 * d["truncated"] / n
        acc       = 100 * d["correct"] / n

        # Storage estimate
        n_target  = 70 if diff == "hard" else 55
        est_gb    = (32 * 4096 * 2 * 100 * avg_rev * n_target) / 1024**3

        total_gb += est_gb
        print(f"\n  {diff.upper()}")
        print(f"    Traces:          {n}")
        print(f"    Avg tokens:      {avg_tok:.0f}")
        print(f"    Avg revisions:   {avg_rev:.1f}")
        print(f"    Accuracy:        {acc:.1f}%")
        print(f"    Truncated:       {trunc_pct:.1f}%")
        print(f"    Est. storage:    {est_gb:.2f} GB")

    print(f"\n  Total est. storage: {total_gb:.2f} GB")

In [ ]:
import shutil
from pathlib import Path
from google.colab import drive
drive.mount("/content/gdrive")
GDRIVE_BASE = Path("/content/gdrive/MyDrive/llm_planning_study")

# Files and folders to delete
to_delete = [
    # Dataset summary — regenerates with new N_HARD/N_MEDIUM
    GDRIVE_BASE / "dataset_summary.json",

    # All trace folders — regenerate with new problem counts
    GDRIVE_BASE / "traces",

    # All label folders — regenerate after new traces
    GDRIVE_BASE / "labels",
    GDRIVE_BASE / "labels_summary_all_models.json",

    # All activation folders — regenerate after new labels
    GDRIVE_BASE / "activations",
    GDRIVE_BASE / "activations_summary_all_models.json",

    # Pilot summary if it exists
    GDRIVE_BASE / "pilot_summary_all_models.json",
]

for path in to_delete:
    if path.exists():
        if path.is_dir():
            shutil.rmtree(path)
            print(f"  Deleted folder: {path.name}")
        else:
            path.unlink()
            print(f"  Deleted file:   {path.name}")
    else:
        print(f"  Not found (ok): {path.name}")

print("\nClean. Ready for fresh run.")